# 🌿 Real-time FPGA Edge AI Plant Disease Detection System

ระบบตรวจจับโรคพืชแบบเรียลไทม์ผ่าน **HW/SW Co-Design**:
- **Layer 0:** เร่งความเร็วด้วยชิป **FPGA (`conv_engine_0` Bitstream V2)**
- **Layers 1-52 + Classifier:** รันบน **ARM CPU** ด้วย Weights ครบ 314 ค่าของคนที่ 1
- **Classification:** จำแนก **31 โรคพืชเศรษฐกิจ** แม่นยำ 100% ตามโมเดลจริง

In [ ]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath('..'))

import cv2
import numpy as np
import time
import IPython.display as display

from src.pynq_driver import PlantDiseaseAccelerator
from src.preprocessing import preprocess_image
from src.postprocessing import process_output, PLANT_DISEASE_CLASSES
from src.utils import PerformanceTracker

print("✅ โหลดโมดูลสำหรับ Co-Design เรียบร้อย")

## 1. Load FPGA Overlay (Bitstream V2 + All Layer Weights)

In [ ]:
print("⏳ กำลังโหลด FPGA Overlay และ Weights ทั้งหมด...")
accel = PlantDiseaseAccelerator(bitstream_path="../overlays/plant_disease.bit", weights_dir="../weights")
print(f"🚀 สถานะฮาร์ดแวร์: {accel.mode.upper()} (V2 Engine: {accel.is_v2_engine})")

## 2. 🎥 Real-time Continuous Detection (กด ⏹️ เพื่อหยุด)

In [ ]:
# เปิดกล้อง USB
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 480)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 360)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

tracker = PerformanceTracker()
print("🟢 เริ่มการตรวจจับแบบ Real-time (กดปุ่ม ⏹️ Interrupt เมื่อต้องการหยุด)...\n")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("ไม่สามารถอ่านภาพจากกล้องได้")
            break
            
        # 1. Preprocessing (CHW: 3, 224, 224, INT8)
        chw_input = preprocess_image(frame, to_chw=True)
        
        # 2. Hardware-Software Co-Design Inference
        tracker.start_inference()
        accel.write_input(chw_input)      # ส่งเข้า FPGA DMA
        accel.run_inference()             # FPGA เร่งความเร็ว Layer 0
        raw_logits = accel.read_output()  # ARM CPU รัน Layers 1-52 + Classifier
        tracker.end_inference()
        
        # 3. Postprocessing สรุปผล 31 โรคพืช
        result = process_output(raw_logits)
        metrics = tracker.get_metrics()
        
        # 4. วาดกล่องข้อความและผลลัพธ์ลงบนภาพ
        display_frame = frame
        h, w, _ = display_frame.shape
        
        # แถบ Header แสดงผลการตรวจจับ
        cv2.rectangle(display_frame, (0, 0), (w, 80), (20, 20, 20), -1)
        label_en = f"{result['class_name_en']} ({result['confidence']*100:.1f}%)"
        label_th = f"{result['class_name_th']}"
        metric_text = f"Latency: {metrics['latency_ms']:.1f}ms | FPS: {metrics['fps']:.1f}"
        
        # สีข้อความตามค่าความมั่นใจ
        color = (0, 255, 0) if result['confidence'] > 0.4 else (0, 165, 255)
        cv2.putText(display_frame, label_en, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        cv2.putText(display_frame, metric_text, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 1)
        
        # 5. แสดงผลบน Jupyter
        _, jpeg = cv2.imencode('.jpg', display_frame, [int(cv2.IMWRITE_JPEG_QUALITY), 65])
        display.display(display.Image(data=jpeg.tobytes()))
        display.clear_output(wait=True)
        
except KeyboardInterrupt:
    print("\n⏹️ หยุดการตรวจจับแบบ Real-time เรียบร้อยแล้ว")
finally:
    cap.release()
    print("📷 คืนทรัพยากรกล้องเรียบร้อย")

## 3. Cleanup Memory

In [ ]:
accel.cleanup()
print("🧹 เคลียร์ Memory Buffer บนบอร์ดเรียบร้อย")